# Intervene on an activation

Compare a policy call before and during a scoped TDHook replacement.

In [ ]:
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.session import HookSession
from tdhook.targets import Target
from xdrl import Interaction

torch.manual_seed(0)
batch = TensorDict({"observation": torch.randn(4, 4)}, batch_size=[4], names=["env"])
policy = TensorDictModule(
    torch.nn.Linear(4, 2, bias=False),
    in_keys=["observation"],
    out_keys=["action"],
)

In [ ]:
interaction = Interaction(policy)
baseline = interaction(batch.clone())["action"].clone()

In [ ]:
target = Target("module", "activation", -1, (0, 1))
with HookSession(interaction.module) as session:
    session.replace(target, 0)
    intervened = interaction(batch.clone())["action"].clone()

assert not torch.equal(baseline, intervened)
assert torch.count_nonzero(intervened) == 0
assert not policy.module._forward_hooks
{
    "baseline_mean": baseline.mean().item(),
    "intervened_mean": intervened.mean().item(),
    "hooks_after_context": len(policy.module._forward_hooks),
}